# Embryo Image Classification - TURBO VERSION

This version is optimized for **speed and performance**.

**Turbo Optimizations:**
1. **MobileNetV2 Architecture**: Extremely fast and efficient.
2. **tf.data API**: Parallel data loading and prefetching (removes CPU bottlenecks).
3. **Mixed Precision (FP16)**: 2x speedup on GPU.
4. **Partial Fine-tuning**: Only unfreezes the top layers to speed up gradient calculations.
5. **160x160 Resolution**: Faster processing with similar accuracy.

In [ ]:
# 1. Setup and Turbo Configuration
import os
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
import matplotlib.pyplot as plt
import zipfile
import requests

In [ ]:
# Enable Mixed Precision for 2x-3x speedup on T4 GPU
try:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("Mixed precision enabled.")
except:
    print("Mixed precision not supported or already set.")

In [ ]:
# 2. Dataset Download & Extract
DATASET_URL = "https://zenodo.org/records/14253170/files/HumanEmbryo2.0.zip?download=1"
ZIP_NAME = "embryo_dataset.zip"

if not os.path.exists(ZIP_NAME):
    print("Downloading dataset...")
    r = requests.get(DATASET_URL, stream=True)
    with open(ZIP_NAME, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024):
            if chunk: f.write(chunk)

with zipfile.ZipFile(ZIP_NAME, 'r') as zip_ref:
    zip_ref.extractall('dataset')

print("Done!")

In [ ]:
# 3. Fast Data Pipeline (tf.data API)
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
DATA_DIR = 'dataset/HumanEmbryo2.0'

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Prefetching and Mapping (CRITICAL FOR SPEED)
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, augment=False):
    # Apply Preprocessing
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    
    if augment:
        augmentation = tf.keras.Sequential([
            layers.RandomRotation(0.2),
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomZoom(0.1),
        ])
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    
    return ds.prefetch(buffer_size=AUTOTUNE)

train_ds = prepare(train_ds, augment=True)
val_ds = prepare(val_ds, augment=False)

In [ ]:
# 4. Build Model
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Start with frozen base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(5, activation='softmax', dtype='float32') # Ensure output is float32 for mixed precision
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Phase 1: Warmup (3 Epochs)
print("Phase 1: Head Warmup...")
model.fit(train_ds, validation_data=val_ds, epochs=3)

In [ ]:
# Phase 2: Partial Fine-tuning (Faster and more stable than full unfreeze)
print("Phase 2: Fine-tuning top blocks...")
base_model.trainable = True

# Unfreeze only the last 20 layers of MobileNetV2
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("embryo_model_turbo.h5", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
from google.colab import files
files.download('embryo_model_turbo.h5')

---
# SECTION: MODEL VALIDATION

This section verifies that the stage classification works correctly on its own by calculating accuracy, visualizing confusion matrices, and inspecting sample predictions and edge cases.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Extract true labels and predictions
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# 2. Accuracy Metrics
acc = accuracy_score(y_true, y_pred_classes)
print(f"Validation Accuracy: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred_classes, target_names=class_names))

In [ ]:
# 3. Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Stage Classification Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# 4. Sample Predictions & Confidence Distribution
plt.figure(figsize=(15, 10))
for images, labels in val_ds.take(1):
    preds = model.predict(images, verbose=0)
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        img_vis = (images[i].numpy() + 1.0) / 2.0
        img_vis = np.clip(img_vis, 0, 1)
        plt.imshow(img_vis)
        true_class = class_names[np.argmax(labels[i])]
        pred_class = class_names[np.argmax(preds[i])]
        conf = np.max(preds[i])
        color = 'green' if true_class == pred_class else 'red'
        plt.title(f"True: {true_class}\nPred: {pred_class} ({conf:.2f})", color=color)
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Edge Case Testing (Simulated Noise / Low Quality)
print("\n--- Edge Case Testing ---")

def add_noise(image):
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.5, dtype=tf.float32)
    return tf.clip_by_value(image + noise, -1.0, 1.0)

noisy_ds = val_ds.map(lambda x, y: (add_noise(x), y))

for images, labels in noisy_ds.take(1):
    preds = model.predict(images, verbose=0)
    plt.figure(figsize=(10, 4))
    for i in range(min(3, len(images))):
        plt.subplot(1, 3, i + 1)
        img_vis = (images[i].numpy() + 1.0) / 2.0
        plt.imshow(np.clip(img_vis, 0, 1))
        true_class = class_names[np.argmax(labels[i])]
        pred_class = class_names[np.argmax(preds[i])]
        conf = np.max(preds[i])
        plt.title(f"Noisy Edge Case\nPred: {pred_class} ({conf:.2f})")
        plt.axis('off')
    plt.show()
    break